# 🧠 Brain Tumor MRI Image Classification
### End-to-End Deep Learning Pipeline with Custom CNN & Transfer Learning

**Author:** Deep Learning Specialist  
**Objective:** Build, train, evaluate, and benchmark deep convolutional neural networks to classify brain MRI scans into 4 clinical categories:
1. **Glioma Tumor**
2. **Meningioma Tumor**
3. **Pituitary Tumor**
4. **No Tumor (Healthy Control)**

## 1. Imports and Environment Setup

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPUs Available: {tf.config.list_physical_devices('GPU')}")

## 2. Dataset Paths and Exploration

In [ ]:
BASE_DIR = 'database'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
VALID_DIR = os.path.join(BASE_DIR, 'valid')
TEST_DIR  = os.path.join(BASE_DIR, 'test')
CLASSES   = ['glioma', 'meningioma', 'no_tumor', 'pituitary']

for split_name, split_path in [('Train', TRAIN_DIR), ('Validation', VALID_DIR), ('Test', TEST_DIR)]:
    print(f"{split_name} Split:")
    total = 0
    for c in CLASSES:
        c_path = os.path.join(split_path, c)
        cnt = len(os.listdir(c_path)) if os.path.exists(c_path) else 0
        total += cnt
        print(f"  - {c}: {cnt} images")
    print(f"  Total = {total} images\n")

## 3. Data Preprocessing & Augmentation Generators

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    horizontal_flip=True,
    zoom_range=0.10,
    width_shift_range=0.08,
    height_shift_range=0.08,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASSES, shuffle=True
)
valid_gen = val_test_datagen.flow_from_directory(
    VALID_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASSES, shuffle=False
)
test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASSES, shuffle=False
)

## 4. Visualizing Sample MRI Scans from Each Category

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, cls_name in enumerate(CLASSES):
    cls_dir = os.path.join(TRAIN_DIR, cls_name)
    first_img = os.listdir(cls_dir)[0]
    img_path = os.path.join(cls_dir, first_img)
    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].set_title(cls_name.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## 5. Model Architecture 1: Custom Deep CNN

In [ ]:
def create_custom_cnn():
    model = models.Sequential([
        layers.Input(shape=(*IMG_SIZE, 3)),
        
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.20),

        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.30),

        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.35),

        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.50),
        layers.Dense(len(CLASSES), activation='softmax')
    ], name='Custom_CNN')
    
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

custom_cnn = create_custom_cnn()
custom_cnn.summary()

## 6. Model Architecture 2: Transfer Learning (MobileNetV2)

In [ ]:
def create_mobilenet_model():
    base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
    base.trainable = False
    
    inputs = layers.Input(shape=(*IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.40)(x)
    outputs = layers.Dense(len(CLASSES), activation='softmax')(x)
    
    model = models.Model(inputs, outputs, name='MobileNetV2_Transfer')
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model, base

mobilenet_model, mobilenet_base = create_mobilenet_model()
mobilenet_model.summary()

## 7. Model Evaluation & Comparison

In [ ]:
# Load pre-computed results or evaluate models
import json
if os.path.exists('model_results.json'):
    with open('model_results.json') as f:
        results = json.load(f)
    print("Loaded Benchmark Results:")
    for m_name, m_stats in results.items():
        print(f"\n{m_name}:")
        print(f"  Accuracy : {m_stats['accuracy']*100:.2f}%")
        print(f"  Precision: {m_stats['precision']*100:.2f}%")
        print(f"  Recall   : {m_stats['recall']*100:.2f}%")
        print(f"  F1-Score : {m_stats['f1_score']*100:.2f}%")

## 8. Real-time Inference Example

In [ ]:
sample_path = 'sample_images/glioma_sample_1.jpg'
if os.path.exists(sample_path):
    img = Image.open(sample_path).convert('RGB')
    img_arr = np.expand_dims(np.array(img.resize(IMG_SIZE)) / 255.0, axis=0)
    
    plt.imshow(img)
    plt.title("Input Brain MRI", fontsize=12)
    plt.axis('off')
    plt.show()
    print(f"Preprocessed Tensor Shape: {img_arr.shape}")